In [1]:
import numpy as np

In [12]:
def softmax(z):
    e_z = np.exp(z)
    return e_z/ e_z.sum(axis = 1, keepdims = True)

def softmax_stable(z):
    e_z = np.exp(z - np.max(z, axis = 1, keepdims = True))
    return e_z / e_z.sum(axis = 1, keepdims = True)

def softmax_loss(X, y, W):
    A = softmax_stable(X.dot(W))
    id0 = range(X.shape[0])
    return -np.mean(np.log(A[id0, y]))

def softmax_grad(X, y, W):
    A = softmax_stable(X.dot(W))
    id0 = range(X.shape[0])
    A[id0, y] -= 1
    return X.T.dot(A)/X.shape[0]

In [15]:
def softmax_fit(X, y, W, lr = 0.01, nepoches = 100, tol = 1e-5, batch_size = 10):
    W_old = W.copy()
    ep = 0
    loss_hist = [softmax_loss(X, y, W)]
    N = X.shape[0]
    nbatches = int(np.ceil(float(N)/batch_size))
    while ep < nepoches:
        ep += 1
        mix_ids = np.random.permutation(N)
        for i in range(nbatches):
            batch_ids = mix_ids[batch_size*i:min(batch_size*(i+1), N)]
            X_batch, y_batch = X[batch_ids], y[batch_ids]
            W -= lr*softmax_grad(X_batch, y_batch, W) # update gradient descent
        loss_hist.append(softmax_loss(X, y, W))
        if np.linalg.norm(W - W_old)/W.size < tol:
            break
        W_old = W.copy()
    return W, loss_hist

def pred(W, X):
    return np.argmax(X.dot(W), axis =1)

In [16]:
C, N = 5, 500
means = [[2, 2], [8, 3], [3, 6], [14, 2], [12, 8]]
cov = [[1, 0], [0, 1]]
X0 = np.random.multivariate_normal(means[0], cov, N)
X1 = np.random.multivariate_normal(means[1], cov, N)
X2 = np.random.multivariate_normal(means[2], cov, N)
X3 = np.random.multivariate_normal(means[3], cov, N)
X4 = np.random.multivariate_normal(means[4], cov, N)
X = np.concatenate((X0, X1, X2, X3, X4), axis = 0)
Xbar = np.concatenate((X, np.ones((X.shape[0], 1))), axis = 1)
y = np.asarray([0]*N + [1]*N + [2]*N+ [3]*N + [4]*N)
W_init = np.random.randn(Xbar.shape[1], C)
W, loss_hist = softmax_fit(Xbar, y, W_init, batch_size = 10, nepoches = 100, lr =
0.05)